In [1]:
import requests
import zipfile
import io
import json
import pandas as pd

# Download the zip from Google Drive
url = "https://drive.google.com/uc?export=download&id=14KV6yKgdjzb6fbFzshGuNvEp9zGv_gol"
response = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(response.content))
z.extractall("feedbackQA_data")

def load_split(filepath):
    with open(filepath, encoding="utf-8") as f:
        fbqa = json.load(f)
    
    rows = []
    for item in fbqa:
        question = item["question"]
        
        # Reconstruct the answer passage
        passage_text = ""
        ref = item["passage"]["reference"]
        if ref.get("page_title"):
            passage_text += "# " + ref["page_title"] + "\n\n"
        if ref.get("section_headers"):
            passage_text += "## " + ", ".join(ref["section_headers"]) + "\n\n"
        if ref.get("section_content"):
            passage_text += ref["section_content"]
        
        rows.append({
            "question": question,
            "passage": passage_text,
            "rating": item["rating"]
        })
    
    return pd.DataFrame(rows)

df = load_split("feedbackQA_data/feedback_valid.json")

In [2]:
rating_map = {
    "Excellent": 1.0,
    "Acceptable": 0.6,
    "Could be Improved": 0.3,
    "Bad": 0.0
}

df["avg_score"] = df["rating"].apply(
    lambda ratings: sum(rating_map[r] for r in ratings) / len(ratings)
)

sample = df.sample(n=100).reset_index(drop=True)
sample

,question,passage,rating,avg_score
0,I work in construction. What guidelines are in...,# Construction and other outdoor work\n\n## 2....,"[Bad, Bad]",0.00
1,What is the purpose of the Business Interrupti...,# Guidance Coronavirus (COVID-19): financial s...,"[Excellent, Excellent]",1.00
2,How can one manage home care in an urban area ...,# Coronavirus (COVID-19) advice for people in ...,"[Bad, Could be Improved]",0.15
3,Will there be a problem with waste management ...,# Clinical Questions about COVID-19: Questions...,"[Acceptable, Bad]",0.30
4,What should students do if they'e not pleased ...,# Guidance Coronavirus (COVID-19): cancellatio...,"[Bad, Bad]",0.00
...,...,...,...,...
95,Who is at risk from COVID-19?,# Frequently Asked Questions\n\n## Higher Risk...,"[Acceptable, Excellent]",0.80
96,What changes can I make to my workplace in ord...,# Labs and research facilities\n\n## 3. Social...,"[Excellent, Excellent]",1.00
97,What is the proof to show children are the chi...,# Guidance Actions for early years and childca...,"[Bad, Bad]",0.00
98,Have any changes been made to WHO guidelines i...,# Q&A: Malaria and COVID-19\n\n## What is WHO ...,"[Excellent, Could be Improved]",0.65


In [3]:
dataset = []

for index, row in sample.iterrows():

    dataset.append(
        {
        "question": row['question'],
        "context": row['passage'],
        "score": row['avg_score']
        }
    )

In [4]:
import json

with open("../../datasets/context.json", "w") as f:
    json.dump(dataset, f)